In [1]:
from cassis.xmi import load_cas_from_xmi
from cassis.typesystem import load_typesystem
from cas_visualizer.visualizer import SpanVisualizer,DependencyVisualizer
from src.crop_example.update_fs import propagate_feature_values
from IPython.display import display, HTML
import ipywidgets as widgets

### Vorbereitung

In [2]:
xmi = open("data/cas_dakoda.xmi").read()
xml = open("data/typesystem_dakoda.xml").read()
cas_org = load_cas_from_xmi(xmi, typesystem=load_typesystem(xml))
print(cas_org.sofa_string)

Über das Thema die finanzielle Entlohnung eines Menschen habe ich schon früher meherere Gedanken gemacht . Die Situation ist in Ungarn hat in den letzten Jahrzehnten nicht genug geändert . Es gibt zu vielen Menschen , die für die Gesellschaft viel geleistet haben , trotzdem bekommen sie dafür wenig Geld . Sie sind zum Beispiel die Ärzte oder die KrankepflegerInnen , die eigentlich für die Gesundheit des Menschen verantwortlich sind und wir sollten ihnen dankbarer wären , dass sie unsere Leben gerettet haben . Aber bezahlt die Staat für ihre Arbeit ganz wenig . Andererseits , wenn man an die Politiker denkt , warum haben sie so höhes Einkommen ? Das habe ich nie verstanden . Es gibt eine ähnliche Situation in dem Fall der Lehrer . Sie müssen neben der Unterricht in der Schule noch auch nachmittags Privatstunden halten , um ihre Entlohnung zu ergänzen . Sie lehren die zukünftige Generationen , aber sie sind trotzdem nicht so entwertet , einen höheren dafür zu bekommen . Deswegen müssen s

/Users/dgardner/GitHub/crop_example/.venv/lib/python3.13/site-packages/cassis/typesystem.py:704: UserWarning: For type [org.dakoda.learnerannotation] feature with name [value] already exists in parent [de.tudarmstadt.ukp.dkpro.core.api.ner.type.NamedEntity]!
  warnings.warn(msg)


In [3]:
# Aktualisiert UDependency-Annotationen mit "begin" und "end" aus Dependent bzw. Governor
cas_org = cas_org.get_view("ctok") # ctok enthält UDependency-Annotationen
propagate_feature_values(cas_org, "UDependency", "end", "Governor")
propagate_feature_values(cas_org, "UDependency", "begin", "Dependent")

# Beispiel: crop_sofa_string

In [4]:
cas_org.sofa_string[107:189]

'Die Situation ist in Ungarn hat in den letzten Jahrzehnten nicht genug geändert . '

In [5]:
cas = cas_org.deep_copy()
cas = cas.get_view('ctok') # ctok enthält UDependency-Annotationen
cas.crop_sofa_string(107, 189, False)
dep_vis = DependencyVisualizer(load_typesystem(xml))

html = dep_vis.visualize(cas)
display(HTML(f'<div style="zoom: 0.4;">{html}</div>'))

# Beispiel: remove_annotations_in_range

In [6]:
cas = cas_org.deep_copy()
cas = cas.get_view('spacy_ctok') # spacy_ctok enthält POS-Annotationen
cas.crop_sofa_string(0, 514, False)
cas.sofa_string

'Über das Thema die finanzielle Entlohnung eines Menschen habe ich schon früher meherere Gedanken gemacht . Die Situation ist in Ungarn hat in den letzten Jahrzehnten nicht genug geändert . Es gibt zu vielen Menschen , die für die Gesellschaft viel geleistet haben , trotzdem bekommen sie dafür wenig Geld . Sie sind zum Beispiel die Ärzte oder die KrankepflegerInnen , die eigentlich für die Gesundheit des Menschen verantwortlich sind und wir sollten ihnen dankbarer wären , dass sie unsere Leben gerettet haben .'

In [7]:
span_vis = SpanVisualizer(load_typesystem(xml))
span_vis.selected_span_type = SpanVisualizer.HIGHLIGHT
span_vis.add_feature(name="de.tudarmstadt.ukp.dkpro.core.api.lexmorph.type.pos.POS", feature="PosValue", value="ADV")
span_vis.add_feature(name="de.tudarmstadt.ukp.dkpro.core.api.lexmorph.type.pos.POS", feature="PosValue", value="ADJA")

html = span_vis.visualize(cas)
display(HTML(html))

In [8]:
cas.remove_annotations_in_range(0,106)
cas.remove_annotations_in_range(190,514)
#Annotationen von "uninteressanten" Textstellen wurden entfernt
html = span_vis.visualize(cas)
display(HTML(html))

# Beispiel: Anzeige einzelner Sätze eines Texts

In [12]:
sentences = []
dep_vis = DependencyVisualizer(load_typesystem(xml))

for annotation in cas_org.get_view("ctok").select("Sentence"):
    tmp_cas = cas_org.deep_copy()
    tmp_cas = tmp_cas.get_view('ctok') # ctok enthält Dependency-Annotationen
    tmp_cas.crop_sofa_string(annotation["begin"], annotation["end"], False)
    html = dep_vis.visualize(tmp_cas)
    # HTML wird in einem div-Container geschrumpft
    scaled_html = f'<div style="zoom: 0.8;">{html}</div>'
    sentences.append(HTML(scaled_html))

In [33]:
slider = widgets.BoundedIntText(value=1, min=1, max=len(sentences), description='Sentence:', layout=widgets.Layout(width='150px'))
total_label = widgets.Label(value=f'of {len(sentences)}', layout=widgets.Layout(margin='2px 40px 0 10px'))
prev_button = widgets.Button(description='\u25C0')
next_button = widgets.Button(description='\u25B6')

def on_prev_clicked(arg):
    if slider.value > slider.min:
        slider.value -= 1

def on_next_clicked(arg):
    if slider.value < slider.max:
        slider.value += 1

prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

ui = widgets.HBox([prev_button, slider, total_label, next_button])

def show_sentence(i):
    display(sentences[i-1])

out = widgets.interactive_output(show_sentence, {'i': slider})

display(ui, out)

Output()